# Simple RAG - Query and Retrieval

**Flow:**

Query → Retrieve → LLM → Answer

## Install Dependencies

In [14]:
# # Conda environment setup
# !pip install langchain langchain-chroma langchain-openai chromadb

## Basic RAG Code Phase 2 - Query and Retrieval

In [15]:
# # Colab setup
# from google.colab import drive
# from google.colab import userdata
# drive.mount('/content/drive')

In [16]:
# # Colab Store Location
# store_location = "/content/drive/MyDrive/rag_langchain/data/chroma_db1"

In [17]:
# # Colab Key
# from google.colab import userdata
# import os
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [18]:
# Local setup
from pathlib import Path

store_location = "../vector_db/chroma_db2"

if not Path(store_location).exists():
    raise FileNotFoundError(f"Vector store not found: {store_location}")

In [19]:
# Use Python dotenv to load environment variables from a .env file
from dotenv import load_dotenv
import os

load_dotenv()

True

### Experiment Config (Set Before Step 1)

Tune this value before running Step 1 so you can quickly test retrieval behavior.


In [20]:
# Recommended default for retrieval in this notebook
RETRIEVAL_K = 6

print("Experiment config:")
print(f"- RETRIEVAL_K: {RETRIEVAL_K}")

Experiment config:
- RETRIEVAL_K: 6


### Step 1: Create retriever

In [21]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedding = OpenAIEmbeddings()

vectorstore = Chroma(
    persist_directory=store_location,
    embedding_function=embedding
)

retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

### Step 2: Build Modern RAG Chain (LCEL)

In [22]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
You are a Security Now RAG assistant.

You are given context retrieved from a Security Now document database.
The corpus contains episode-based sources (for example sn-1072), document types
(transcript/show_notes), and optional page numbers.

Instructions:
- Use ONLY the provided context to answer the question.
- Do NOT use outside knowledge.
- If the answer is not in the context, say: "Not found in provided context."
- Do not make up information.
- For each key claim, cite episode/doc metadata when available.
- Prefer show_notes for concise factual recall and transcripts for detailed explanation.

Citation format:
- Include citations inline using this format: [episode_id | doc_type | page if available]
- Example: [sn-1072 | show_notes | p25]

Context:
{context}

Question:
{question}

Answer:
""")

### Step 3: Compose the chain

In [23]:
import re
from pathlib import Path

def resolve_metadata(raw_metadata: dict) -> dict:
    source = raw_metadata.get("source", "unknown")
    page = raw_metadata.get("page")

    file_name = raw_metadata.get("file_name")
    episode_id = raw_metadata.get("episode_id")
    episode_number = raw_metadata.get("episode_number")
    doc_type = raw_metadata.get("doc_type")

    source_path = Path(source) if source and source != "unknown" else None

    if not file_name and source_path:
        file_name = source_path.name

    if not episode_id and source_path:
        match = re.search(r"(sn-\d+)", source_path.stem.lower())
        if match:
            episode_id = match.group(1)

    if not episode_number and episode_id:
        try:
            episode_number = int(episode_id.split("-")[1])
        except (IndexError, ValueError):
            episode_number = "unknown"

    if not doc_type and source_path:
        suffix = source_path.suffix.lower()
        if suffix == ".txt":
            doc_type = "transcript"
        elif suffix == ".pdf":
            doc_type = "show_notes"

    return {
        "source": source,
        "page": page,
        "file_name": file_name or "unknown",
        "episode_id": episode_id or "unknown",
        "episode_number": episode_number if episode_number is not None else "unknown",
        "doc_type": doc_type or "unknown",
    }

def format_docs(docs):
    formatted = []
    for doc in docs:
        md = resolve_metadata(doc.metadata)
        source = md["source"]
        file_name = md["file_name"]
        episode_id = md["episode_id"]
        episode_number = md["episode_number"]
        doc_type = md["doc_type"]
        page = md["page"]
        location = f"{source} (page {page})" if page else source
        metadata_header = (
            f"Episode: {episode_id} | Episode Number: {episode_number} | "
            f"Doc Type: {doc_type} | File: {file_name} | Location: {location}"
        )
        formatted.append(f"{metadata_header}\n{doc.page_content}")
    return "\n\n".join(formatted)

def extract_episode_from_question(question: str) -> str | None:
    match = re.search(r"\b(sn-\d+)\b", question.lower())
    return match.group(1) if match else None

def retrieve_docs(question: str):
    episode_id = extract_episode_from_question(question)

    if episode_id:
        candidate_k = max(RETRIEVAL_K * 5, 20)
        candidates = vectorstore.similarity_search(question, k=candidate_k)

        episode_docs = []
        for d in candidates:
            md = resolve_metadata(d.metadata)
            source = str(md.get("source", "")).lower()
            meta_episode = str(md.get("episode_id", "")).lower()
            if meta_episode == episode_id or episode_id in source:
                episode_docs.append(d)

        if episode_docs:
            return episode_docs[:RETRIEVAL_K]

    return retriever.invoke(question)

rag_chain = (
    {
        "context": RunnableLambda(retrieve_docs) | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Step 4: Query

In [27]:
# Set up multiple questions
questions = [
    "What are the main topics across these documents?",
    "any microsoft security news?",
    "any CVE 10 security incidents?",
    "Can summarize episode sn-1069?",
    "Can summarize episode sn-1070?",
    "Summarize the most recent episode with CVE 10 security incidents."
]

In [28]:
for idx, question in enumerate(questions, start=1):
    response = rag_chain.invoke(question)
    print("=" * 60)
    print(f"Question {idx}: {question}")
    print("-" * 60)
    print(response)
    print()

Question 1: What are the main topics across these documents?
------------------------------------------------------------
The main topics across the provided documents include:

1. **Security Features in macOS**: The intercept dialog in macOS Terminal that warns users about pasting suspicious text, indicating potential malware risks. This feature aims to protect users from scams that encourage harmful actions through Terminal commands [sn-1072 | transcript | p1] and [sn-1072 | show_notes | p11].

2. **User Experiences with Cyber Attacks**: A listener's account of a cyber attack involving the remote installation of a software program, Screen Connect, on their PC, which led to unauthorized money transfers. This highlights the risks of remote access tools and the importance of user awareness [sn-1061 | show_notes | p14].

3. **Cultural Commentary on Media**: Discussions about the Apple TV series "For All Mankind," which presents an alternative history of the space race, reflecting on its 

## Debug

In [29]:
for idx, question in enumerate(questions, start=1):
    print("=" * 60)
    print(f"Debug for Question {idx}: {question}")
    print("=" * 60)

    docs = retrieve_docs(question)
    for i, d in enumerate(docs, start=1):
        md = resolve_metadata(d.metadata)
        source = md["source"]
        file_name = md["file_name"]
        episode_id = md["episode_id"]
        episode_number = md["episode_number"]
        doc_type = md["doc_type"]
        page = md["page"]
        location = f"{source} (page {page})" if page else source
        print(
            f"chunk {i} | episode={episode_id} ({episode_number}) | "
            f"type={doc_type} | file={file_name} | location={location}"
        )
        print(d.page_content)
        print("-" * 50)

    print()

Debug for Question 1: What are the main topics across these documents?
chunk 1 | episode=sn-1072 (1072) | type=transcript | file=sn-1072.txt | location=../data/sn-1072.txt
Terminal, an intercept dialog will be displayed to caution the user about the possible implications of what they are attempting to do.  The dialog reads:  "Possible malware, Paste blocked."  And it says:  "Your Mac has not been harmed.  Scammers often encourage pasting text into Terminal to try and harm your Mac or compromise your privacy.  These instructions are commonly offered via websites, chat agents, apps, files, or a phone call."
--------------------------------------------------
chunk 2 | episode=sn-1072 (1072) | type=show_notes | file=sn-1072-notes.pdf | location=../data/sn-1072-notes.pdf (page 11)
macOS user of Terminal attempts to paste a suspicious string into 
Terminal, an intercept dialog will be displayed to caution the user 
about the possible implications of what they are attempting to do. 
The dialo